# Start here: what is a neural network?

One neuron, then a layer, then a tiny network, all with plain Python and a little PyTorch so that every number can be checked by hand.

## One neuron

Multiply each input by its weight, add the bias, then bend the total with an activation function.

In [1]:
x = [2.0, 1.0, 3.0]            # the inputs
w = [0.5, -1.0, 0.25]          # one weight for each input
bias = 0.5

total = sum(wi * xi for wi, xi in zip(w, x)) + bias
output = max(0.0, total)       # ReLU: keep positive totals, zero the rest
print("total:", total, " output after ReLU:", output)

total: 1.25  output after ReLU: 1.25


## Change the numbers and watch the output

The weights and the bias are the neuron's memory.

In [2]:
def neuron(x, w, bias):
    total = sum(wi * xi for wi, xi in zip(w, x)) + bias
    return max(0.0, total)

print("original      :", neuron(x, [0.5, -1.0, 0.25], 0.5))
print("first weight 2:", neuron(x, [2.0, -1.0, 0.25], 0.5))
print("bias -2       :", neuron(x, [0.5, -1.0, 0.25], -2.0))

original      : 1.25
first weight 2: 4.25
bias -2       : 0.0


## A layer is several neurons that share the same inputs

The three neurons below are the hidden layer in Figure 2 of the learning module. Written as a matrix product, the layer is one line.

In [3]:
import torch

x = torch.tensor([2.0, 1.0, 3.0])
W = torch.tensor([[0.5, -1.0, 0.25],     # neuron 1 (the one from above)
                  [-0.3, 0.8, 0.1],      # neuron 2
                  [0.2, 0.2, -0.4]])     # neuron 3
B = torch.tensor([0.5, -0.2, 0.0])

before_relu = W @ x + B
hidden = torch.relu(before_relu)
print("before ReLU:", before_relu.round(decimals=2))
print("hidden layer output:", hidden.round(decimals=2))

before ReLU: 

tensor([ 1.2500,  0.3000, -0.6000])
hidden layer output: tensor([1.2500, 0.3000, 0.0000])


## The output neuron turns the hidden values into a probability

In [4]:
W2 = torch.tensor([[1.0, -0.5, 0.8]])
B2 = torch.tensor([-0.1])
score = W2 @ hidden + B2
print("score:", round(score.item(), 3))
print("probability:", round(torch.sigmoid(score).item(), 3))

score: 1.0
probability: 0.731


## The same network as an nn.Module

PyTorch builds the same thing from layers. The weights are random until we copy ours in, and random weights give an arbitrary answer. Training is the process that replaces random numbers with useful ones.

In [5]:
from torch import nn

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(3, 3), nn.ReLU(), nn.Linear(3, 1), nn.Sigmoid())
print("random weights give:", round(net(x).item(), 3))
print("parameters:", sum(p.numel() for p in net.parameters()), " = (3*3 + 3) + (3*1 + 1)")

with torch.no_grad():                      # copy in the weights from above
    net[0].weight.copy_(W); net[0].bias.copy_(B)
    net[2].weight.copy_(W2); net[2].bias.copy_(B2)
print("our weights give   :", round(net(x).item(), 3))

random weights give: 0.495
parameters: 16  = (3*3 + 3) + (3*1 + 1)
our weights give   : 0.731


## Your turn

1. Change the input to `[0.0, 0.0, 0.0]`. Which neurons still give a non-zero answer, and why?
2. Make the output weight on the third hidden neuron very large. Does it change the answer? Why not?
3. Add a fourth hidden neuron. How many parameters does the network have now?